# CNN baseline (no attention)

Trains the **plain** guided-SR CNN (`model.cnn.GuidedCNN`) on the new
4 km → 1 km MODIS LST downscaling task. Same dual-branch architecture
as the attention model but with vanilla residual blocks — establishes
how much of the gain (if any) comes from attention vs. the architecture.


## 0 — Setup

In [ ]:
# Colab: install deps. Local: skip if already installed.
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q rasterio geopandas h5py huggingface_hub scikit-learn


In [ ]:
# Pull the model + dataloader code. On Colab we clone; locally we assume cwd is the repo.
import os, sys
if IN_COLAB:
    if not os.path.isdir('downscaling'):
        !git clone -q https://github.com/fresleven/downscaling.git
    %cd downscaling
    sys.path.insert(0, '.')
else:
    # Add repo root to path (notebook lives in notebooks/colabs/)
    sys.path.insert(0, os.path.abspath('../..'))


In [ ]:
from huggingface_hub import snapshot_download
DATA_ROOT = 'data'
snapshot_download(
    repo_id='akhot2/downscaling',
    repo_type='dataset',
    local_dir=DATA_ROOT,
)


## 1 — Datasets

In [ ]:
from model.dataset import DownscalingDataset, get_dataloaders

train_ds = DownscalingDataset(root=DATA_ROOT, split='train', download=False)
val_ds   = DownscalingDataset(root=DATA_ROOT, split='val',   download=False)
test_ds  = DownscalingDataset(root=DATA_ROOT, split='test',  download=False)

print(f'Samples — train: {len(train_ds)} ({len(train_ds.dates)} dates × {len(train_ds.split_blocks)} blocks)')
print(f'           val:  {len(val_ds)},  test: {len(test_ds)}')
print(f'HR shape: {train_ds.hr_shape},  LR shape: {train_ds.lr_shape}')
print(f'LULC classes ({len(train_ds.lulc_classes)}): {train_ds.lulc_classes.tolist()}')


## 2 — Training utilities

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=0)


def make_hr_cov(batch):
    return torch.cat([batch['ndvi'], batch['dem'], batch['lulc_onehot'], batch['loc']], dim=1)


def masked_mse(pred, target, mask):
    diff2 = (pred - target) ** 2 * mask
    return diff2.sum() / mask.sum().clamp(min=1)


@torch.no_grad()
def eval_loader(model, loader):
    model.eval()
    rmses, maes = [], []
    for b in loader:
        b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
        pred = model(b['lr_lst'], make_hr_cov(b))
        for i in range(pred.shape[0]):
            m = b['valid_mask'][i, 0].bool()
            if not m.any():
                continue
            d = pred[i, 0][m] - b['hr_lst'][i, 0][m]
            rmses.append(torch.sqrt((d ** 2).mean()).item())
            maes.append(d.abs().mean().item())
    return {'RMSE': float(np.mean(rmses)), 'MAE': float(np.mean(maes))}


def train_model(model, n_epochs=60, lr=1e-3, weight_decay=1e-5):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    history, best = [], {'val_rmse': float('inf'), 'state': None, 'epoch': -1}
    for epoch in range(n_epochs):
        model.train()
        losses = []
        for b in train_loader:
            b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
            pred = model(b['lr_lst'], make_hr_cov(b))
            loss = masked_mse(pred, b['hr_lst'], b['valid_mask'])
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        sched.step()
        v = eval_loader(model, val_loader)
        history.append({'epoch': epoch, 'train_mse': float(np.mean(losses)), **v})
        if v['RMSE'] < best['val_rmse']:
            best = {'val_rmse': v['RMSE'],
                    'state': {k: t.detach().cpu().clone() for k, t in model.state_dict().items()},
                    'epoch': epoch}
        if epoch % 5 == 0 or epoch == n_epochs - 1:
            print(f"epoch {epoch:3d}  train_mse={np.mean(losses):.3f}  "
                  f"val RMSE={v['RMSE']:.3f}  MAE={v['MAE']:.3f}  (best @ {best['epoch']})")
    model.load_state_dict(best['state'])
    return model, history, best


## 3 — Build & train

In [ ]:
from model.cnn import GuidedCNN

n_classes = train_ds.lulc_classes.shape[0]
cov_channels = 1 + 1 + n_classes + 2  # NDVI + DEM + LULC one-hot + loc (x, y)

model = GuidedCNN(cov_channels=cov_channels, base_channels=64,
                  n_lr_blocks=4, n_hr_blocks=6)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

model, history, best = train_model(model, n_epochs=60, lr=1e-3)
print(f"\nBest val RMSE = {best['val_rmse']:.3f}°C @ epoch {best['epoch']}")


## 4 — Loss curves

In [ ]:
import matplotlib.pyplot as plt
ep = [h['epoch'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, [h['train_mse'] for h in history], label='train MSE'); ax[0].grid(); ax[0].legend()
ax[1].plot(ep, [h['RMSE'] for h in history], label='val RMSE'); ax[1].grid(); ax[1].legend()
ax[1].set_xlabel('epoch'); plt.show()


## 5 — Test evaluation & sample predictions

In [ ]:
test_metrics = eval_loader(model, test_loader)
print('Test:', test_metrics)

import matplotlib.pyplot as plt

# --- Sample block predictions ---
batch = next(iter(test_loader))
b_dev = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
with torch.no_grad():
    pred = model(b_dev['lr_lst'], make_hr_cov(b_dev)).cpu()

n_show = min(4, pred.shape[0])
fig, axes = plt.subplots(3, n_show, figsize=(3*n_show, 7))
if n_show == 1: axes = axes[:, None]
for i in range(n_show):
    hr = batch['hr_lst'][i, 0].numpy()
    lr = batch['lr_lst'][i, 0].numpy()
    pr = pred[i, 0].numpy()
    vmin = float(np.nanpercentile(hr, 2)); vmax = float(np.nanpercentile(hr, 98))
    for r, (img, lbl) in enumerate([(lr, 'LR input'), (pr, 'Prediction'), (hr, 'HR truth')]):
        ax = axes[r, i]; ax.imshow(img, cmap='RdYlBu_r', vmin=vmin, vmax=vmax)
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0: ax.set_ylabel(lbl)
    axes[0, i].set_title(f"{batch['date'][i]}\nblock {batch['block_id'][i].item()}", fontsize=8)
plt.tight_layout(); plt.show()

# --- RMSE per elevation band ---
elev_all, err2_all = [], []
model.eval()
with torch.no_grad():
    for b in test_loader:
        b_dev = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
        pr = model(b_dev['lr_lst'], make_hr_cov(b_dev)).cpu()
        for i in range(pr.shape[0]):
            m = b['valid_mask'][i, 0].bool()
            if not m.any(): continue
            elev_all.append(b['dem'][i, 0][m].numpy())
            err2_all.append((pr[i, 0][m] - b['hr_lst'][i, 0][m]).numpy() ** 2)

elev_all = np.concatenate(elev_all)
err2_all = np.concatenate(err2_all)

n_bins = 8
edges = np.percentile(elev_all, np.linspace(0, 100, n_bins + 1))
labels, rmses = [], []
for j in range(n_bins):
    sel = (elev_all >= edges[j]) & (elev_all <= edges[j + 1])
    if sel.sum() > 0:
        labels.append(f'{edges[j]:.0f}–{edges[j+1]:.0f} m')
        rmses.append(float(np.sqrt(err2_all[sel].mean())))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(labels)), rmses, color='steelblue', edgecolor='black', linewidth=0.4)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_xlabel('Elevation band'); ax.set_ylabel('RMSE (°C)')
ax.set_title('Test RMSE by elevation band')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('rmse_by_elevation.png', dpi=150)
plt.show()
print('Saved rmse_by_elevation.png')


## 6 — Save weights

In [ ]:
torch.save(model.state_dict(), 'cnn_baseline.pt')
print('Saved cnn_baseline.pt')
